# Process Reward Modeling Improves Multi-Step Reasoning Stability in Small Language Models

**Research Question:** Do PRMs improve reasoning consistency across different prompts?

**Experiments:**
1. Train a Process Reward Model (PRM) on reasoning traces
2. Evaluate on GSM8K, MATH, and ARC benchmarks
3. Measure consistency: same question, different phrasings → same answer?

---

## Setup

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets accelerate peft trl bitsandbytes wandb \
    numpy pandas scikit-learn matplotlib seaborn tqdm jsonlines sentencepiece evaluate scipy

In [ ]:
# Clone the repo (update with your GitHub username)
!git clone https://github.com/sanowl/Process-Reward-Modeling-Improves-Multi-Step-Reasoning-Stability-in-Small-Language-Models.git
%cd Process-Reward-Modeling-Improves-Multi-Step-Reasoning-Stability-in-Small-Language-Models

In [ ]:
import torch
import numpy as np
import json
import os
from tqdm.notebook import tqdm

from src.config import ModelConfig, PRMConfig, TrainingConfig, EvalConfig, DataConfig
from src.models.base_model import load_base_model, load_tokenizer
from src.models.prm import ProcessRewardModel, StepLabelDataset, PRMTrainer
from src.data.dataset_loader import load_benchmark_dataset, load_prm_training_data
from src.data.reasoning_traces import generate_reasoning_traces, parse_reasoning_steps, label_reasoning_steps
from src.data.paraphraser import generate_paraphrases, generate_paraphrase_dataset
from src.evaluation.consistency import ConsistencyEvaluator
from src.evaluation.benchmarks import BenchmarkEvaluator
from src.evaluation.best_of_n import BestOfNSelector
from src.visualization.plots import (
    plot_consistency_comparison, plot_accuracy_comparison, plot_step_scores
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Load Base Model & Datasets

In [ ]:
# Configuration - adjust model size based on your GPU
# Options: "microsoft/phi-2" (2.7B), "Qwen/Qwen2.5-1.5B", "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MODEL_NAME = "microsoft/phi-2"
MAX_SAMPLES = 50  # Start small for testing, increase for real experiments

model_config = ModelConfig(
    model_name=MODEL_NAME,
    use_4bit=True,
    use_lora=True,
)

print("Loading model...")
model = load_base_model(model_config)
tokenizer = load_tokenizer(model_config)
print("Model loaded!")

In [ ]:
# Load benchmark datasets
print("Loading datasets...")
gsm8k = load_benchmark_dataset("gsm8k", "test", max_samples=MAX_SAMPLES)
print(f"GSM8K test: {len(gsm8k)} examples")

math_data = load_benchmark_dataset("math", "test", max_samples=MAX_SAMPLES)
print(f"MATH test: {len(math_data)} examples")

arc_data = load_benchmark_dataset("arc", "test", max_samples=MAX_SAMPLES)
print(f"ARC test: {len(arc_data)} examples")

# Preview
print("\n--- GSM8K Example ---")
print(f"Q: {gsm8k[0]['question'][:200]}...")
print(f"A: {gsm8k[0]['answer']}")

## 2. Generate Reasoning Traces

In [ ]:
# Generate reasoning traces for a few examples
sample_questions = gsm8k["question"][:5]

print("Generating reasoning traces...")
traces = generate_reasoning_traces(
    model, tokenizer, sample_questions,
    num_samples=3,
    temperature=0.7,
    device=device,
)

# Show an example trace
print(f"\n--- Question: {sample_questions[0][:100]}... ---")
for i, trace in enumerate(traces[0]):
    print(f"\nTrace {i+1}:")
    for j, step in enumerate(trace['steps']):
        print(f"  Step {j+1}: {step[:80]}...")
    print(f"  Final answer: {trace['final_answer']}")

## 3. Train Process Reward Model

In [ ]:
# Option A: Load PRM800K training data (if available)
# prm_data = load_prm_training_data(max_samples=1000)

# Option B: Generate our own training data from model traces
print("Generating PRM training data from model traces...")
train_questions = gsm8k["question"]
train_answers = gsm8k["answer"]

prm_traces = []
for q, gold in tqdm(zip(train_questions, train_answers), total=len(train_questions)):
    question_traces = generate_reasoning_traces(
        model, tokenizer, [q],
        num_samples=4,
        temperature=0.7,
        device=device,
    )[0]
    
    for trace in question_traces:
        step_labels = label_reasoning_steps(trace['steps'], gold, trace['final_answer'])
        steps, labels = zip(*step_labels) if step_labels else ([], [])
        prm_traces.append({
            "steps": list(steps),
            "step_labels": list(labels),
        })

print(f"Generated {len(prm_traces)} training traces")

In [ ]:
# Train the PRM
prm_config = PRMConfig(
    reward_model_name=MODEL_NAME,
    hidden_size=2560,  # Match phi-2; change for other models
)

training_config = TrainingConfig(
    num_epochs=3,
    batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
)

# Prepare datasets
split_idx = int(0.9 * len(prm_traces))
train_dataset = StepLabelDataset(prm_traces[:split_idx], tokenizer)
eval_dataset = StepLabelDataset(prm_traces[split_idx:], tokenizer)

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

# Initialize PRM
prm = ProcessRewardModel(prm_config)

# Train
trainer = PRMTrainer(prm, train_dataset, eval_dataset, training_config)
trained_prm = trainer.train()

# Save checkpoint
os.makedirs("outputs/prm", exist_ok=True)
torch.save(trained_prm.state_dict(), "outputs/prm/prm_model.pt")
print("PRM training complete!")

## 4. Visualize PRM Step Scores

In [ ]:
# Score a reasoning trace with the trained PRM
prm.eval()
sample_trace = traces[0][0]  # First trace of first question

step_scores = prm.score_steps(
    sample_trace['full_text'], tokenizer, device=device
)

print("Step scores:")
for i, (step, score) in enumerate(zip(sample_trace['steps'], step_scores)):
    print(f"  Step {i+1} (score={score:.3f}): {step[:60]}...")

plot_step_scores(
    step_scores,
    title="PRM Scores for Sample Reasoning Trace",
    save_path="outputs/step_scores_example.png"
)

## 5. Experiment: Reasoning Consistency

**Core experiment:** For each question, generate answers using different prompt phrasings.
Compare consistency with and without PRM-guided selection.

In [ ]:
# Run consistency experiment
NUM_PARAPHRASES = 5
BEST_OF_N = 8
EVAL_SAMPLES = 30  # Increase for final results

evaluator = ConsistencyEvaluator()
selector = BestOfNSelector(prm, tokenizer, aggregation="min", device=device)

baseline_all = []  # per-question list of answers (baseline)
prm_all = []       # per-question list of answers (PRM)

for idx in tqdm(range(min(EVAL_SAMPLES, len(gsm8k)))):
    question = gsm8k[idx]["question"]
    prompts = generate_paraphrases(question, "gsm8k", NUM_PARAPHRASES, seed=idx)
    
    baseline_answers = []
    prm_answers = []
    
    for prompt in prompts:
        traces = generate_reasoning_traces(
            model, tokenizer, [prompt],
            num_samples=BEST_OF_N,
            temperature=0.7,
            device=device,
        )[0]
        
        # Baseline: majority vote
        answers = [t["final_answer"] for t in traces]
        from collections import Counter
        majority = Counter(answers).most_common(1)[0][0]
        baseline_answers.append(majority)
        
        # PRM: best-of-N
        trace_texts = [t["full_text"] for t in traces]
        best_idx, scores = selector.select_best(trace_texts)
        prm_answers.append(traces[best_idx]["final_answer"])
    
    baseline_all.append(baseline_answers)
    prm_all.append(prm_answers)

print("\nDone generating answers!")

In [ ]:
# Compute and display consistency metrics
baseline_metrics = evaluator.compute_consistency_metrics(baseline_all)
prm_metrics = evaluator.compute_consistency_metrics(prm_all)

print("=" * 50)
print("CONSISTENCY RESULTS (GSM8K)")
print("=" * 50)
print(f"\n{'Metric':<30} {'Baseline':>10} {'PRM':>10} {'Delta':>10}")
print("-" * 60)
for metric in ["mean_agreement_rate", "mean_majority_fraction", "mean_entropy", "perfect_consistency_rate"]:
    b = baseline_metrics[metric]
    p = prm_metrics[metric]
    d = p - b
    print(f"{metric:<30} {b:>10.4f} {p:>10.4f} {d:>+10.4f}")

## 6. Experiment: Accuracy (Baseline vs PRM)

In [ ]:
# Accuracy evaluation
bench_eval = BenchmarkEvaluator(model, tokenizer, device=device)

print("Evaluating greedy accuracy on GSM8K...")
greedy_results = bench_eval.evaluate_accuracy(gsm8k, max_samples=EVAL_SAMPLES, temperature=0.0)
print(f"Greedy accuracy: {greedy_results['accuracy']:.4f}")

print("\nEvaluating majority vote accuracy...")
majority_results = bench_eval.evaluate_accuracy(
    gsm8k, max_samples=EVAL_SAMPLES, num_samples=BEST_OF_N, temperature=0.7
)
print(f"Majority vote accuracy: {majority_results['accuracy']:.4f}")

print("\nEvaluating PRM best-of-N accuracy...")
prm_results = bench_eval.evaluate_with_prm(
    gsm8k, prm, tokenizer,
    num_samples=BEST_OF_N, max_samples=EVAL_SAMPLES
)
print(f"PRM best-of-N accuracy: {prm_results['accuracy']:.4f}")

## 7. Generate Plots

In [ ]:
# Consistency comparison plot
consistency_results = {
    "gsm8k": {
        "baseline": baseline_metrics,
        "prm": prm_metrics,
    }
}

plot_consistency_comparison(
    consistency_results,
    save_path="outputs/consistency_comparison.png"
)

In [ ]:
# Accuracy comparison plot
accuracy_results = {
    "gsm8k": {
        "greedy": {"accuracy": greedy_results["accuracy"]},
        "majority_vote": {"accuracy": majority_results["accuracy"]},
        "prm_best_of_n": {"accuracy": prm_results["accuracy"]},
    }
}

plot_accuracy_comparison(
    accuracy_results,
    save_path="outputs/accuracy_comparison.png"
)

## 8. Save Results

In [ ]:
# Save all results
os.makedirs("results", exist_ok=True)

all_results = {
    "model": MODEL_NAME,
    "num_paraphrases": NUM_PARAPHRASES,
    "best_of_n": BEST_OF_N,
    "eval_samples": EVAL_SAMPLES,
    "consistency": {
        "baseline": baseline_metrics,
        "prm": prm_metrics,
    },
    "accuracy": {
        "greedy": greedy_results["accuracy"],
        "majority_vote": majority_results["accuracy"],
        "prm_best_of_n": prm_results["accuracy"],
    },
}

with open("results/experiment_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)

print("Results saved to results/experiment_results.json")
print("\n" + "=" * 50)
print("EXPERIMENT COMPLETE")
print("=" * 50)

## Next Steps

To extend to full paper results:

1. **Scale up**: Set `MAX_SAMPLES = None` and `EVAL_SAMPLES = None` to run on full datasets
2. **More benchmarks**: Add MATH and ARC using `run_experiments.py`
3. **More models**: Try `Qwen/Qwen2.5-1.5B`, `TinyLlama/TinyLlama-1.1B-Chat-v1.0`
4. **Ablations**: Vary `BEST_OF_N` (4, 8, 16, 32) and `NUM_PARAPHRASES` (3, 5, 8)
5. **PRM aggregation**: Compare `min` vs `mean` vs `last` vs `product`

```bash
# Full experiment from command line:
python scripts/train_prm.py --model_name microsoft/phi-2 --output_dir outputs/prm
python scripts/run_experiments.py --model_name microsoft/phi-2 --prm_checkpoint outputs/prm/prm_model.pt
```